In [2]:
!pip install deep-translator yt-dlp gensim
!pip install git+https://github.com/openai/whisper.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.2/172.2 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 20.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.2
    Uninstalling scipy-1.15.2:
      Successfully uninstalled scipy-1.15.2
ERROR: pip's dependency resolver does not currently take into accoun

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
from multiprocessing import Pool, Manager
import whisper
import yt_dlp
import numpy as np
import re
import pandas as pd
from deep_translator import GoogleTranslator
from tensorflow.keras.models import load_model
from gensim.models import FastText

model_path = "/content/drive/MyDrive/Fitra_RL_Models/CNN_LSTM_reinforced.h5"
fasttext_path = "/content/drive/MyDrive/Fitra_RL_Models/fasttext_model.model"

# تحميل النماذج
model = load_model(model_path)
fasttext_model = FastText.load(fasttext_path)


def prepare_input(title_tokens=None, desc_tokens=None, transcript_tokens=None):
    def get_vector(tokens):
        vectors = [fasttext_model.wv[word] for word in tokens if word in fasttext_model.wv]
        return np.mean(vectors, axis=0) if vectors else np.zeros(300)

    title_vector = get_vector(title_tokens) if title_tokens else np.zeros(300)
    desc_vector = get_vector(desc_tokens) if desc_tokens else np.zeros(300)
    transcript_vector = get_vector(transcript_tokens) if transcript_tokens else np.zeros(300)

    combined = np.hstack([title_vector, desc_vector, transcript_vector])
    return combined.reshape(1, 900, 1)


def detect_language_with_whisper(audio_path):
    temp_model = whisper.load_model("tiny")
    result = temp_model.transcribe(audio_path, fp16=False, language=None)
    return result.get("language", "en")


def extract_transcript(video_url, lock):
    audio_path = "audio.mp3"
    ydl_opts = {'format': 'bestaudio', 'outtmpl': audio_path}

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_url])
    except Exception as e:
        print(f"❌ Failed to download audio from: {video_url} \nReason: {e}")
        return None

    detected_lang = detect_language_with_whisper(audio_path)
    print(f"🔤 اللغة المكتشفة: {detected_lang}")

    if detected_lang == "ar":
        with lock:
            print("🔒 دخول القسم الحرج: استخدام Whisper Medium...")
            dynamic_model = whisper.load_model("medium")
            result = dynamic_model.transcribe(audio_path, fp16=False, language=detected_lang)
    else:
        dynamic_model = whisper.load_model("base")
        result = dynamic_model.transcribe(audio_path, fp16=False, language=detected_lang)

    os.remove(audio_path)
    return result["text"]


def is_arabic(text):
    if not isinstance(text, str):
        return False
    for char in text:
        if '\u0600' <= char <= '\u06FF':
            return True
    return False

def get_video_metadata(video_url):
    ydl_opts = {
        'quiet': True,
        'skip_download': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)
        title = info.get('title', '')
        description = info.get('description', '')
        return title, description

def analyze_video(video_url):
    print("⬇️ جاري استخراج الترانسكربت...")
    transcript = extract_transcript(video_url)

    if transcript is None:
        print("⚠️ تم تخطي هذا الفيديو لعدم توفره.")
        return "غير معروف"

    print("\n📝 الترانسكربت:\n", transcript[:500])

    lang = "ar" if is_arabic(transcript) else "en"
    print(f"🌐 اللغة المحددة للتحليل: {lang}")

    if lang == "en":
        title, description = get_video_metadata(video_url)
        def tokenize(text):
            return re.sub(r"[^\w\s]", "", text).lower().split() if isinstance(text, str) else []
        title_tokens = tokenize(title)
        desc_tokens = tokenize(description)
    else:
        title_tokens = []
        desc_tokens = []

    transcript_tokens = re.sub(r"[^\w\s]", "", transcript).lower().split()
    x_input = prepare_input(title_tokens, desc_tokens, transcript_tokens)
    prediction = model.predict(x_input)[0][0]
    result = "شاذ" if prediction > 0.5 else "غير شاذ"
    print(f"✅ النتيجة: {result} (احتمالية = {prediction:.2f})")
    return result


  def process_video(args):
    url, idx, total, lock = args
    print(f"\n🔎 Analyzing video {idx+1}/{total}: {url}")

    transcript = extract_transcript(url, lock)
    if transcript is None:
        print("⚠️ تم تخطي هذا الفيديو.")
        return

    lang = "ar" if is_arabic(transcript) else "en"
    print(f"🌐 اللغة المحددة للتحليل: {lang}")

    if lang == "en":
        title, description = get_video_metadata(url)
        def tokenize(text):
            return re.sub(r"[^\w\s]", "", text).lower().split() if isinstance(text, str) else []
        title_tokens = tokenize(title)
        desc_tokens = tokenize(description)
    else:
        title_tokens = []
        desc_tokens = []

    transcript_tokens = re.sub(r"[^\w\s]", "", transcript).lower().split()
    x_input = prepare_input(title_tokens, desc_tokens, transcript_tokens)
    prediction = model.predict(x_input)[0][0]
    result = "شاذ" if prediction > 0.5 else "غير شاذ"
    print(f"✅ النتيجة: {result} (احتمالية = {prediction:.2f})")


def process_videos_parallel(excel_file, num_processes=None):
    if num_processes is None:
        num_processes = os.cpu_count()

    df = pd.read_excel(excel_file)
    if 'video_url' not in df.columns:
        print("❌ The Excel file does not contain a 'video_url' column.")
        return

    video_urls = df['video_url'].tolist()

    manager = Manager()
    lock = manager.Lock()  # 🔒 lock للقسم الحرج عند استخدام Whisper Medium

    args = [(url, idx, len(video_urls), lock) for idx, url in enumerate(video_urls)]

    print(f"🚀 Starting parallel processing with {num_processes} processes...\n")

    with Pool(processes=num_processes) as pool:
        pool.map(process_video, args)

    print("\n✅ Completed processing all videos.")

  if __name__ == "__main__":
    process_videos_parallel('/content/channel_videos_8.xlsx')





In [4]:
process_videos_until_flagged('/content/channel_videos_8 (1).xlsx')

🔎 Analyzing video 1: https://youtu.be/KytE8zKxciE?si=2WI-7VSOU4Mt736W
⬇️ جاري استخراج الترانسكربت...
[youtube] Extracting URL: https://youtu.be/KytE8zKxciE?si=2WI-7VSOU4Mt736W
[youtube] KytE8zKxciE: Downloading webpage
[youtube] KytE8zKxciE: Downloading tv client config
[youtube] KytE8zKxciE: Downloading player fded239a-main
[youtube] KytE8zKxciE: Downloading tv player API JSON
[youtube] KytE8zKxciE: Downloading ios player API JSON
[youtube] KytE8zKxciE: Downloading m3u8 information
[info] KytE8zKxciE: Downloading 1 format(s): 251
[download] Destination: audio.mp3
[download] 100% of    1.64MiB in 00:00:00 at 2.63MiB/s   
🔤 اللغة المكتشفة: ar - استخدام موديل Whisper: medium


100%|█████████████████████████████████████| 1.42G/1.42G [00:55<00:00, 27.5MiB/s]



📝 الترانسكربت:
  أهلا بكم أصدقائي في قناة أبجد اليوم سوف نتعلم جدول الدرب أربع هيا بنا نبدأ أربع ضرب واحد يساوي أربع أربع ضرب اثنان يساوي ثمانية أربع ضرب ثلاثة يساوي اثنى عشر أربع ضرب أربع يساوي ستة عشر أربع ضرب خمسة يساوي عشرون أربع ضرب ستة يساوي أربع وعشرون أربع ضرب ستة ضرب سبعة يساوي ثمانين عشرون أربع ضرب ثمانية يساوي اثنان وثلاثون أربع ضرب تسعة يساوي ستة وثلاثون أربع ضرب واحد يساوي أربع أربع ضرب اثنان يساوي ثمانية أربع ضرب ثلاثة يساوي اثنى عشر أربع ضرب أربع يساوي ستة عشر أربع ضرب خمسة يساوي عشرون أربع ضرب س
🌐 اللغة المحددة للتحليل: ar
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
✅ النتيجة: غير شاذ (احتمالية = 0.15)
🔎 Analyzing video 2: https://youtu.be/3S2b1Ju7W6Q?si=IqJO6wev_vj4yURW
⬇️ جاري استخراج الترانسكربت...
[youtube] Extracting URL: https://youtu.be/3S2b1Ju7W6Q?si=IqJO6wev_vj4yURW
[youtube] 3S2b1Ju7W6Q: Downloading webpage
[youtube] 3S2b1Ju7W6Q: Downloading tv client config
[youtube] 3S2b1Ju7W6Q: Downloading player fded239a-main
[youtube] 3S2b1Ju7W6Q: Downloading tv player API JSON


###Chaneels


In [ ]:
import yt_dlp
import pandas as pd
import os

def fetch_channel_video_links(channel_input, output_excel='channel_videos.xlsx'):
    video_links = []

    # إذا المستخدم أعطى معرّف فقط، نحوله لرابط قناة
    if channel_input.startswith('UC') and len(channel_input) > 20:
        channel_url = f"https://www.youtube.com/channel/{channel_input}/videos"
    elif channel_input.startswith('https://'):
        channel_url = channel_input
        if not channel_url.endswith('/videos'):
            if channel_url.endswith('/'):
                channel_url += 'videos'
            else:
                channel_url += '/videos'
    else:
        print("❌ Invalid input. Provide a full YouTube channel URL or channel ID starting with 'UC'.")
        return

    options = {
        'quiet': True,
        'extract_flat': True,
        'skip_download': True,
    }

    with yt_dlp.YoutubeDL(options) as ydl:
        try:
            info = ydl.extract_info(channel_url, download=False)
            if info is None or 'entries' not in info:
                print("❌ No videos found in the channel.")
                return

            for entry in info['entries']:
                if entry and 'url' in entry:
                    video_id = entry['url']
                    video_url = f"https://www.youtube.com/watch?v={video_id}"
                    video_links.append(video_url)

            if video_links:
                df = pd.DataFrame({'video_url': video_links})
                df.to_excel(output_excel, index=False)
                full_path = os.path.abspath(output_excel)
                print(f"✅ Saved {len(video_links)} videos to:\n{full_path}")
            else:
                print("❌ No video links extracted.")

        except Exception as e:
            print(f"❌ Error: {e}")


In [ ]:
fetch_channel_video_links('UC543IxLnWvmFhhNTH50vCaw')


✅ Saved 314 videos to:
/content/channel_videos.xlsx


In [ ]:

import pandas as pd

def clean_video_urls(input_excel, output_excel):
    # Read the Excel file
    df = pd.read_excel(input_excel)

    if 'video_url' not in df.columns:
        print("❌ The Excel file does not contain a 'video_url' column. Please check the file.")
        return

    cleaned_urls = []

    for url in df['video_url']:
        if pd.isna(url):
            continue  # Skip empty cells

        real_url = str(url).strip()  # لا نقطع، فقط ننظف المسافات

        cleaned_urls.append(real_url)

    # Create a cleaned DataFrame
    cleaned_df = pd.DataFrame({'video_url': cleaned_urls})

    # Save it to a new Excel file
    cleaned_df.to_excel(output_excel, index=False)
    print(f"✅ Cleaned URLs saved to: {output_excel}")


In [ ]:
clean_video_urls('channel_videos.xlsx', 'channel_videos_cleaned.xlsx')


✅ Cleaned URLs saved to: channel_videos_cleaned.xlsx


##Parallel code

In [1]:
!pip install deep-translator yt-dlp gensim
!pip install git+https://github.com/openai/whisper.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.2/172.2 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 14.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.2
    Uninstalling scipy-1.15.2:
      Successfully uninstalled scipy-1.15.2
ERROR: pip's dependency resolver does not currently take into accoun

In [ ]:
!pip install deep-translator yt-dlp gensim
!pip install git+https://github.com/openai/whisper.git

In [ ]:
import os
from multiprocessing import Pool, Manager
import whisper
import yt_dlp
import numpy as np
import re
import pandas as pd
from deep_translator import GoogleTranslator
from tensorflow.keras.models import load_model
from gensim.models import FastText

# Model paths
model_path = "/content/drive/MyDrive/Fitra_RL_Models/CNN_LSTM_reinforced.h5"
fasttext_path = "/content/drive/MyDrive/Fitra_RL_Models/fasttext_model.model"

# Load models
model = load_model(model_path)
fasttext_model = FastText.load(fasttext_path)

def prepare_input(title_tokens=None, desc_tokens=None, transcript_tokens=None):
    def get_vector(tokens):
        vectors = [fasttext_model.wv[word] for word in tokens if word in fasttext_model.wv]
        return np.mean(vectors, axis=0) if vectors else np.zeros(300)

    title_vector = get_vector(title_tokens) if title_tokens else np.zeros(300)
    desc_vector = get_vector(desc_tokens) if desc_tokens else np.zeros(300)
    transcript_vector = get_vector(transcript_tokens) if transcript_tokens else np.zeros(300)

    combined = np.hstack([title_vector, desc_vector, transcript_vector])
    return combined.reshape(1, 900, 1)

def detect_language_with_whisper(audio_path):
    temp_model = whisper.load_model("tiny")
    result = temp_model.transcribe(audio_path, fp16=False, language=None)
    return result.get("language", "en")

def extract_transcript(video_url, lock):
    audio_path = f"audio_{os.getpid()}_{hash(video_url)%100000}.mp3"  # Unique filename per video/process
    ydl_opts = {'format': 'bestaudio', 'outtmpl': audio_path}

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_url])
    except Exception as e:
        print(f"❌ Failed to download audio from: {video_url}\nReason: {e}")
        return None

    detected_lang = detect_language_with_whisper(audio_path)
    print(f"🔤 Detected language: {detected_lang}")

    try:
        if detected_lang == "ar":
            with lock:
                print("🔒 Entering critical section: Using Whisper Medium...")
                dynamic_model = whisper.load_model("medium")
                result = dynamic_model.transcribe(audio_path, fp16=False, language=detected_lang)
        else:
            dynamic_model = whisper.load_model("base")
            result = dynamic_model.transcribe(audio_path, fp16=False, language=detected_lang)
    finally:
        if os.path.exists(audio_path):
            os.remove(audio_path)  # 🗑 Delete audio after transcription

    return result["text"]

def is_arabic(text):
    if not isinstance(text, str):
        return False
    for char in text:
        if '\u0600' <= char <= '\u06FF':
            return True
    return False

def get_video_metadata(video_url):
    ydl_opts = {
        'quiet': True,
        'skip_download': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)
        title = info.get('title', '')
        description = info.get('description', '')
        return title, description

def process_video(args):
    url, idx, total, lock = args
    print(f"\n🔎 Analyzing video {idx+1}/{total}: {url}")

    transcript = extract_transcript(url, lock)
    if transcript is None:
        print("⚠️ Skipped due to missing transcript.")
        return

    lang = "ar" if is_arabic(transcript) else "en"
    print(f"🌍 Language used for analysis: {lang}")

    if lang == "en":
        title, description = get_video_metadata(url)
        def tokenize(text):
            return re.sub(r"[^\w\s]", "", text).lower().split() if isinstance(text, str) else []
        title_tokens = tokenize(title)
        desc_tokens = tokenize(description)
    else:
        title_tokens = []
        desc_tokens = []

    transcript_tokens = re.sub(r"[^\w\s]", "", transcript).lower().split()
    x_input = prepare_input(title_tokens, desc_tokens, transcript_tokens)
    prediction = model.predict(x_input)[0][0]
    result = "Inappropriate" if prediction > 0.5 else "Safe"
    print(f"✅ Result: {result} (Probability = {prediction:.2f})")

def process_videos_parallel(excel_file, num_processes=None):
    if num_processes is None:
        num_processes = os.cpu_count()

    df = pd.read_excel(excel_file)
    if 'video_url' not in df.columns:
        print("❌ The Excel file does not contain a 'video_url' column.")
        return

    video_urls = df['video_url'].tolist()
    manager = Manager()
    lock = manager.Lock()

    args = [(url, idx, len(video_urls), lock) for idx, url in enumerate(video_urls)]

    print(f"🚀 Starting parallel analysis with {num_processes} processes...\n")

    with Pool(processes=num_processes) as pool:
        pool.map(process_video, args)

    print("\n✅ All videos have been processed.")

if __name__ == "__main__":
    process_videos_parallel('/content/channel_videos_8 .xlsx')


🚀 Starting parallel analysis with 2 processes...


🔎 Analyzing video 1/4: https://youtu.be/KytE8zKxciE?si=2WI-7VSOU4Mt736W

🔎 Analyzing video 2/4: https://youtu.be/3S2b1Ju7W6Q?si=IqJO6wev_vj4yURW
[youtube] Extracting URL: https://youtu.be/KytE8zKxciE?si=2WI-7VSOU4Mt736W
[youtube] KytE8zKxciE: Downloading webpage
[youtube] Extracting URL: https://youtu.be/3S2b1Ju7W6Q?si=IqJO6wev_vj4yURW
[youtube] 3S2b1Ju7W6Q: Downloading webpage
[youtube] KytE8zKxciE: Downloading tv client config
[youtube] KytE8zKxciE: Downloading player fded239a-main
[youtube] KytE8zKxciE: Downloading tv player API JSON
[youtube] 3S2b1Ju7W6Q: Downloading tv client config
[youtube] 3S2b1Ju7W6Q: Downloading player fded239a-main
[youtube] KytE8zKxciE: Downloading ios player API JSON
[youtube] 3S2b1Ju7W6Q: Downloading tv player API JSON
[youtube] 3S2b1Ju7W6Q: Downloading ios player API JSON
[youtube] KytE8zKxciE: Downloading m3u8 information
[youtube] 3S2b1Ju7W6Q: Downloading m3u8 information
[info] KytE8zKxciE: Downloadin